# ClimateTwinBench — Benchmark Generator
Generates 100 benchmark questions (50 Trend Analysis + 50 Comparative Reasoning)
from ClimateTwinIndia dataset (climate_states_2019.csv … climate_states_2025.csv)

In [1]:
import pandas as pd
import numpy as np
import json
import os
from scipy import stats

## 1. Load & Merge CSVs

In [3]:
dfs = []

for year in range(2019, 2026):
    path = f"../data/processed/climate_states_{year}.csv"
    if not os.path.exists(path):
        print(f"  WARNING: {path} not found — skipping")
        continue
    tmp = pd.read_csv(path)
    if 'year' not in tmp.columns:
        tmp['year'] = year
    dfs.append(tmp)
    print(f"  Loaded {path}: {len(tmp):,} rows")

df = pd.concat(dfs, ignore_index=True)

print(f"\nMerged shape : {df.shape}")
print(f"Columns      : {df.columns.tolist()}")

  Loaded ../data/processed/climate_states_2019.csv: 4,964 rows
  Loaded ../data/processed/climate_states_2020.csv: 4,964 rows
  Loaded ../data/processed/climate_states_2021.csv: 4,964 rows
  Loaded ../data/processed/climate_states_2022.csv: 4,964 rows
  Loaded ../data/processed/climate_states_2023.csv: 4,964 rows
  Loaded ../data/processed/climate_states_2024.csv: 4,964 rows
  Loaded ../data/processed/climate_states_2025.csv: 4,964 rows

Merged shape : (34748, 8)
Columns      : ['date', 'latitude', 'longitude', 'rainfall_mm', 'max_temperature_c', 'completeness_score', 'anomaly_score', 'year']


In [4]:
print(df.head(3).to_string())
print("\nNull counts:")
print(df.isnull().sum())

   date  latitude  longitude  rainfall_mm  max_temperature_c  completeness_score  anomaly_score  year
0  2019     36.75      77.00     3.331672          31.718817                 1.0      -0.737311  2019
1  2019     36.75      77.25     3.166381          31.718817                 1.0      -0.313557  2019
2  2019     36.75      77.50     2.666646          33.467411                 1.0       0.373344  2019

Null counts:
date                  0
latitude              0
longitude             0
rainfall_mm           0
max_temperature_c     0
completeness_score    0
anomaly_score         0
year                  0
dtype: int64


## 2. Prepare Cell-Level Time Series

In [5]:
# Risk labels
df['risk_label'] = np.where(df['anomaly_score'].abs() >= 10, 'EXTREME',
                    np.where(df['anomaly_score'].abs() >= 5,  'HIGH',
                    np.where(df['anomaly_score'].abs() >= 2,  'MODERATE', 'LOW')))

print(df['risk_label'].value_counts())

risk_label
LOW         33841
MODERATE      856
HIGH           42
EXTREME         9
Name: count, dtype: int64


In [6]:
# Build one dict per (lat, lon) with 7-year time series
# keys: years, rainfall_mm, max_temperature_c, anomaly_score, risk_label

cell_series = {}

for (lat, lon), grp in df.groupby(['latitude', 'longitude']):
    grp = grp.sort_values('year')
    cell_series[(lat, lon)] = {
        'years'            : grp['year'].tolist(),
        'rainfall_mm'      : grp['rainfall_mm'].tolist(),
        'max_temperature_c': grp['max_temperature_c'].tolist(),
        'anomaly_score'    : grp['anomaly_score'].tolist(),
        'risk_label'       : grp['risk_label'].tolist(),
    }

cells = list(cell_series.keys())
years = sorted(df['year'].unique().tolist())
print(f"Unique cells : {len(cells):,}")
print(f"Years        : {years}")

Unique cells : 4,964
Years        : [2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [7]:
# Yearly aggregate stats (used for comparative questions)
yearly_mean_anomaly = df.groupby('year')['anomaly_score'].mean().to_dict()
yearly_mean_rain    = df.groupby('year')['rainfall_mm'].mean().to_dict()
yearly_mean_temp    = df.groupby('year')['max_temperature_c'].mean().to_dict()
yearly_max_anomaly  = df.groupby('year')['anomaly_score'].apply(lambda x: x.abs().max()).to_dict()

print("Yearly mean anomaly_score:")
for y, v in yearly_mean_anomaly.items():
    print(f"  {y}: {v:.4f}")

Yearly mean anomaly_score:
  2019: 0.1354
  2020: 0.1464
  2021: 0.0152
  2022: 0.1030
  2023: -0.3792
  2024: -0.0208
  2025: 0.1327


## 3. Generate 50 Trend Analysis Questions

In [8]:
rng = np.random.default_rng(42)
benchmark = []
ta_counter = 0

# Sample 50 random cells for trend analysis questions
sampled_idxs = rng.choice(len(cells), size=min(50, len(cells)), replace=False)
ta_cells     = [cells[i] for i in sampled_idxs]

print(f"Selected {len(ta_cells)} cells for trend analysis questions")

Selected 50 cells for trend analysis questions


### TA-1 to TA-10 — Rainfall Trend Direction (MCQ)

In [9]:
for lat, lon in ta_cells[:10]:
    s    = cell_series[(lat, lon)]
    vals = s['rainfall_mm']
    yrs  = s['years']

    slope, _, r, p, _ = stats.linregress(range(len(vals)), vals)

    if p > 0.05:
        answer  = 'C'
        explain = f'Slope={slope:.4f} but p={p:.4f} > 0.05 — not significant'
    elif slope > 0:
        answer  = 'A'
        explain = f'Slope={slope:.4f} (p={p:.4f}) — significant increasing trend'
    else:
        answer  = 'B'
        explain = f'Slope={slope:.4f} (p={p:.4f}) — significant decreasing trend'

    vals_str = ', '.join(f'{v:.2f}' for v in vals)
    ta_counter += 1

    benchmark.append({
        'id'         : f'TA-{ta_counter:03d}',
        'category'   : 'trend_analysis',
        'subcategory': 'rainfall_trend_direction',
        'difficulty' : 'easy' if abs(slope) > 0.5 else 'medium',
        'question'   : (
            f'Annual rainfall_mm for grid cell ({lat:.2f}°N, {lon:.2f}°E), '
            f'{yrs[0]}–{yrs[-1]}:\n{vals_str}\n\n'
            f'Which best describes the rainfall trend?'
        ),
        'options': {
            'A': 'Increasing (positive slope, statistically significant)',
            'B': 'Decreasing (negative slope, statistically significant)',
            'C': 'No significant trend (p > 0.05)',
        },
        'answer'     : answer,
        'explanation': explain,
        'context': {
            'latitude' : lat, 'longitude': lon,
            'years'    : yrs, 'rainfall_mm': [round(v,4) for v in vals],
            'slope'    : round(slope, 6), 'p_value': round(p, 6),
        }
    })

print(f'Generated TA-001 to TA-{ta_counter:03d}')

Generated TA-001 to TA-010


### TA-11 to TA-20 — Peak Anomaly Year (MCQ)

In [10]:
for lat, lon in ta_cells[10:20]:
    s         = cell_series[(lat, lon)]
    anomalies = s['anomaly_score']
    yrs       = s['years']

    peak_idx  = int(np.argmax(np.abs(anomalies)))
    peak_year = yrs[peak_idx]
    peak_val  = round(anomalies[peak_idx], 4)

    # 3 distractor years (not the correct one)
    other_years = [y for y in yrs if y != peak_year]
    rng.shuffle(other_years)
    distractors = other_years[:3]

    all_opts = [peak_year] + distractors
    rng.shuffle(all_opts)
    options  = {chr(65+i): str(v) for i, v in enumerate(all_opts)}
    ans_key  = next(k for k, v in options.items() if int(v) == peak_year)

    sorted_abs = sorted(np.abs(anomalies), reverse=True)
    gap        = sorted_abs[0] - sorted_abs[1] if len(sorted_abs) > 1 else 99
    difficulty = 'easy' if gap > 2 else ('medium' if gap > 0.5 else 'hard')

    anom_str = ', '.join(f'{y}:{round(v,2)}' for y, v in zip(yrs, anomalies))
    ta_counter += 1

    benchmark.append({
        'id'         : f'TA-{ta_counter:03d}',
        'category'   : 'trend_analysis',
        'subcategory': 'peak_anomaly_year',
        'difficulty' : difficulty,
        'question'   : (
            f'Anomaly scores for ({lat:.2f}°N, {lon:.2f}°E):\n{anom_str}\n\n'
            f'In which year was the anomaly_score magnitude (|value|) highest?'
        ),
        'options'    : options,
        'answer'     : ans_key,
        'explanation': f'|anomaly| peaks at {peak_year} with value {peak_val} (gap to 2nd={gap:.4f})',
        'context': {
            'latitude'     : lat, 'longitude': lon,
            'years'        : yrs,
            'anomaly_score': [round(v,4) for v in anomalies],
            'peak_year'    : peak_year, 'peak_value': peak_val,
        }
    })

print(f'Generated TA-011 to TA-{ta_counter:03d}')

Generated TA-011 to TA-020


### TA-21 to TA-30 — Total Rainfall Change 2019→2025 (Numerical MCQ)

In [11]:
for lat, lon in ta_cells[20:30]:
    s    = cell_series[(lat, lon)]
    vals = s['rainfall_mm']
    yrs  = s['years']

    if yrs[0] not in yrs or yrs[-1] not in yrs:
        continue

    first_val  = vals[0]
    last_val   = vals[-1]
    total_chg  = round(last_val - first_val, 3)

    # Plausible distractors
    d1 = round(total_chg * 0.6, 3)
    d2 = round(total_chg * 1.4, 3)
    d3 = round(total_chg + (vals[1] - vals[0]), 3)

    all_opts = [str(total_chg), str(d1), str(d2), str(d3)]
    rng.shuffle(all_opts)
    options  = {chr(65+i): v for i, v in enumerate(all_opts)}
    ans_key  = next(k for k, v in options.items() if float(v) == total_chg)

    difficulty = 'easy' if abs(total_chg) > 5 else 'medium'
    vals_str   = ', '.join(f'{v:.2f}' for v in vals)
    ta_counter += 1

    benchmark.append({
        'id'         : f'TA-{ta_counter:03d}',
        'category'   : 'trend_analysis',
        'subcategory': 'total_rainfall_change',
        'difficulty' : difficulty,
        'question'   : (
            f'Annual rainfall_mm for ({lat:.2f}°N, {lon:.2f}°E):\n{vals_str}\n\n'
            f'What is the total change in rainfall_mm from {yrs[0]} to {yrs[-1]} '
            f'(final value minus initial value)?'
        ),
        'options'    : options,
        'answer'     : ans_key,
        'explanation': f'{last_val:.3f} − {first_val:.3f} = {total_chg}',
        'context': {
            'latitude'    : lat, 'longitude': lon,
            'years'       : yrs, 'rainfall_mm': [round(v,4) for v in vals],
            'first_value' : round(first_val, 4), 'last_value': round(last_val, 4),
            'total_change': total_chg,
        }
    })

print(f'Generated TA-021 to TA-{ta_counter:03d}')

Generated TA-021 to TA-030


### TA-31 to TA-40 — Rainfall vs Anomaly Same Direction? (Binary MCQ)

In [12]:
for lat, lon in ta_cells[30:40]:
    s      = cell_series[(lat, lon)]
    r_vals = s['rainfall_mm']
    a_vals = s['anomaly_score']
    yrs    = s['years']
    x      = list(range(len(yrs)))

    r_slope, _, _, r_p, _ = stats.linregress(x, r_vals)
    a_slope, _, _, a_p, _ = stats.linregress(x, a_vals)

    r_sig = r_p < 0.05
    a_sig = a_p < 0.05

    if not r_sig or not a_sig:
        answer  = 'C'
        explain = 'One or both trends are not statistically significant (p > 0.05)'
    elif (r_slope > 0) == (a_slope > 0):
        answer  = 'A'
        explain = f'Both increasing: rain slope={r_slope:.4f}, anomaly slope={a_slope:.4f}' if r_slope > 0 else \
                  f'Both decreasing: rain slope={r_slope:.4f}, anomaly slope={a_slope:.4f}'
    else:
        answer  = 'B'
        explain = f'Opposite: rain slope={r_slope:.4f}, anomaly slope={a_slope:.4f}'

    r_str  = ', '.join(f'{v:.2f}' for v in r_vals)
    a_str  = ', '.join(f'{v:.2f}' for v in a_vals)
    ta_counter += 1

    benchmark.append({
        'id'         : f'TA-{ta_counter:03d}',
        'category'   : 'trend_analysis',
        'subcategory': 'multivariable_direction_agreement',
        'difficulty' : 'medium',
        'question'   : (
            f'For grid cell ({lat:.2f}°N, {lon:.2f}°E), {yrs[0]}–{yrs[-1]}:\n'
            f'  rainfall_mm   : {r_str}\n'
            f'  anomaly_score : {a_str}\n\n'
            f'Do rainfall_mm and anomaly_score trend in the same direction?'
        ),
        'options': {
            'A': 'Yes — both trend in the same direction (both up or both down)',
            'B': 'No — they trend in opposite directions',
            'C': 'Cannot determine — one or both trends are not statistically significant',
        },
        'answer'     : answer,
        'explanation': explain,
        'context': {
            'latitude'       : lat, 'longitude': lon, 'years': yrs,
            'rainfall_mm'    : [round(v,4) for v in r_vals],
            'anomaly_score'  : [round(v,4) for v in a_vals],
            'rainfall_slope' : round(r_slope, 6), 'rainfall_p': round(r_p, 6),
            'anomaly_slope'  : round(a_slope, 6), 'anomaly_p': round(a_p, 6),
        }
    })

print(f'Generated TA-031 to TA-{ta_counter:03d}')

Generated TA-031 to TA-040


### TA-41 to TA-50 — Regime Shift Detection (MCQ)

In [13]:
for lat, lon in ta_cells[40:50]:
    s       = cell_series[(lat, lon)]
    anomaly = s['anomaly_score']
    yrs     = s['years']

    mid = len(anomaly) // 2
    h1  = anomaly[:mid]
    h2  = anomaly[mid:]
    m1, m2 = np.mean(h1), np.mean(h2)
    delta   = m2 - m1

    _, p = stats.mannwhitneyu(h1, h2, alternative='two-sided')

    if p > 0.10 or abs(delta) < 1.0:
        answer  = 'C'
        explain = f'MannWhitney p={p:.4f}, |delta|={abs(delta):.4f} — no significant shift'
    elif delta < 0:
        answer  = 'A'
        explain = f'H1 mean={m1:.3f} → H2 mean={m2:.3f}: wet-to-dry (MWU p={p:.4f})'
    else:
        answer  = 'B'
        explain = f'H1 mean={m1:.3f} → H2 mean={m2:.3f}: dry-to-wet (MWU p={p:.4f})'

    anom_str = '\n'.join(f'  {y}: {round(v,2):+.2f}' for y, v in zip(yrs, anomaly))
    ta_counter += 1

    benchmark.append({
        'id'         : f'TA-{ta_counter:03d}',
        'category'   : 'trend_analysis',
        'subcategory': 'regime_shift_detection',
        'difficulty' : 'hard',
        'question'   : (
            f'Annual anomaly_score for ({lat:.2f}°N, {lon:.2f}°E):\n{anom_str}\n\n'
            f'Which best characterises the long-term anomaly behaviour?'
        ),
        'options': {
            'A': 'Wet-to-dry shift: anomalies move from positive to negative over the period',
            'B': 'Dry-to-wet shift: anomalies move from negative to positive over the period',
            'C': 'No significant regime shift detected',
        },
        'answer'     : answer,
        'explanation': explain,
        'context': {
            'latitude'     : lat, 'longitude': lon, 'years': yrs,
            'anomaly_score': [round(v,4) for v in anomaly],
            'first_half_mean' : round(float(m1), 6),
            'second_half_mean': round(float(m2), 6),
            'delta'        : round(float(delta), 6),
            'mannwhitney_p': round(float(p), 6),
        }
    })

print(f'Generated TA-041 to TA-{ta_counter:03d}')
print(f'Total trend analysis questions so far: {ta_counter}')

Generated TA-041 to TA-050
Total trend analysis questions so far: 50


## 4. Generate 50 Comparative Reasoning Questions

In [14]:
cr_counter = 0

# Sample cell pairs for cross-cell comparisons
all_idxs   = rng.choice(len(cells), size=min(100, len(cells)), replace=False)
cr_cells   = [cells[i] for i in all_idxs]

print(f'Selected {len(cr_cells)} cells for comparative reasoning questions')

Selected 100 cells for comparative reasoning questions


### CR-01 to CR-15 — Cross-Cell Rainfall Comparison for a Single Year (MCQ)

In [15]:
# Pair up cells and compare rainfall in one year
pair_idxs = list(range(0, 30, 2))  # pairs: (0,1), (2,3), ..., (28,29)

for i in pair_idxs[:15]:
    lat1, lon1 = cr_cells[i]
    lat2, lon2 = cr_cells[i + 1]

    s1 = cell_series[(lat1, lon1)]
    s2 = cell_series[(lat2, lon2)]

    # Pick a year both cells have data for
    common_years = [y for y in s1['years'] if y in s2['years']]
    if not common_years:
        continue
    target_year = int(rng.choice(common_years))

    v1 = round(s1['rainfall_mm'][s1['years'].index(target_year)], 3)
    v2 = round(s2['rainfall_mm'][s2['years'].index(target_year)], 3)
    diff = abs(v1 - v2)

    if diff < 0.05:
        answer = 'C'
        explain = f'Values approximately equal: {v1} vs {v2}'
    elif v1 > v2:
        answer = 'A'
        explain = f'Cell A={v1:.3f} > Cell B={v2:.3f} (diff={diff:.3f})'
    else:
        answer = 'B'
        explain = f'Cell B={v2:.3f} > Cell A={v1:.3f} (diff={diff:.3f})'

    difficulty = 'easy' if diff > 5 else ('medium' if diff > 1 else 'hard')
    cr_counter += 1

    benchmark.append({
        'id'         : f'CR-{cr_counter:03d}',
        'category'   : 'comparative_reasoning',
        'subcategory': 'cross_cell_rainfall_comparison',
        'difficulty' : difficulty,
        'question'   : (
            f'In {target_year}, which grid cell had higher rainfall_mm?\n\n'
            f'Cell A: ({lat1:.2f}°N, {lon1:.2f}°E) — rainfall_mm = {v1}\n'
            f'Cell B: ({lat2:.2f}°N, {lon2:.2f}°E) — rainfall_mm = {v2}'
        ),
        'options': {
            'A': f'Cell A ({lat1:.2f}°N, {lon1:.2f}°E)',
            'B': f'Cell B ({lat2:.2f}°N, {lon2:.2f}°E)',
            'C': 'They are approximately equal',
        },
        'answer'     : answer,
        'explanation': explain,
        'context': {
            'year'        : target_year,
            'cell_a'      : {'latitude': lat1, 'longitude': lon1, 'rainfall_mm': v1},
            'cell_b'      : {'latitude': lat2, 'longitude': lon2, 'rainfall_mm': v2},
            'difference'  : round(diff, 4),
        }
    })

print(f'Generated CR-001 to CR-{cr_counter:03d}')

Generated CR-001 to CR-015


### CR-16 to CR-30 — Within-Cell Year Comparison: Which Year Was More Anomalous? (MCQ)

In [16]:
for lat, lon in cr_cells[30:45]:
    s       = cell_series[(lat, lon)]
    anomaly = s['anomaly_score']
    yrs     = s['years']

    if len(yrs) < 2:
        continue

    # Pick two different years
    yi1, yi2 = rng.choice(len(yrs), size=2, replace=False)
    y1, y2   = yrs[int(yi1)], yrs[int(yi2)]
    a1, a2   = round(anomaly[int(yi1)], 4), round(anomaly[int(yi2)], 4)
    diff     = abs(abs(a1) - abs(a2))

    if diff < 0.05:
        answer  = 'C'
        explain = f'|{y1}|={abs(a1):.4f} ≈ |{y2}|={abs(a2):.4f}'
    elif abs(a1) > abs(a2):
        answer  = 'A'
        explain = f'|anomaly in {y1}|={abs(a1):.4f} > |anomaly in {y2}|={abs(a2):.4f}'
    else:
        answer  = 'B'
        explain = f'|anomaly in {y2}|={abs(a2):.4f} > |anomaly in {y1}|={abs(a1):.4f}'

    difficulty = 'easy' if diff > 2 else ('medium' if diff > 0.5 else 'hard')
    cr_counter += 1

    benchmark.append({
        'id'         : f'CR-{cr_counter:03d}',
        'category'   : 'comparative_reasoning',
        'subcategory': 'within_cell_year_anomaly_comparison',
        'difficulty' : difficulty,
        'question'   : (
            f'For grid cell ({lat:.2f}°N, {lon:.2f}°E):\n'
            f'  {y1} anomaly_score = {a1}\n'
            f'  {y2} anomaly_score = {a2}\n\n'
            f'Which year had the more severe climate anomaly (higher |anomaly_score|)?'
        ),
        'options': {
            'A': f'{y1} was more anomalous (|{a1}| > |{a2}|)',
            'B': f'{y2} was more anomalous (|{a2}| > |{a1}|)',
            'C': 'Both years had approximately equal anomaly severity',
        },
        'answer'     : answer,
        'explanation': explain,
        'context': {
            'latitude'    : lat, 'longitude': lon,
            'year1'       : y1, 'anomaly1': a1, 'abs_anomaly1': round(abs(a1),4),
            'year2'       : y2, 'anomaly2': a2, 'abs_anomaly2': round(abs(a2),4),
            'difference'  : round(diff, 4),
        }
    })

print(f'Generated CR-016 to CR-{cr_counter:03d}')

Generated CR-016 to CR-030


### CR-31 to CR-40 — Global Year Ranking: Which Year Was Most/Least Anomalous? (MCQ)

In [17]:
# Questions about global yearly statistics
yr_table = '\n'.join(
    f'  {y}: mean anomaly={yearly_mean_anomaly[y]:.4f}, '
    f'mean rainfall={yearly_mean_rain[y]:.4f}, '
    f'max |anomaly|={yearly_max_anomaly[y]:.4f}'
    for y in years
)

global_questions = [
    {
        'q'  : 'Which year had the highest mean anomaly_score across all grid cells?',
        'key': 'mean_anomaly_max',
        'val': max(yearly_mean_anomaly, key=yearly_mean_anomaly.get),
        'exp': f'Yearly mean anomalies: {{"+".join(str(k)+":"+str(round(v,4)) for k,v in yearly_mean_anomaly.items())}}',
    },
    {
        'q'  : 'Which year had the lowest mean anomaly_score across all grid cells?',
        'key': 'mean_anomaly_min',
        'val': min(yearly_mean_anomaly, key=yearly_mean_anomaly.get),
        'exp': f'Yearly mean anomalies ranked ascending',
    },
    {
        'q'  : 'Which year had the highest maximum |anomaly_score| in any single grid cell?',
        'key': 'max_abs_anomaly',
        'val': max(yearly_max_anomaly, key=yearly_max_anomaly.get),
        'exp': f'Yearly max |anomaly|: {{"+".join(str(k)+":"+str(round(v,4)) for k,v in yearly_max_anomaly.items())}}',
    },
    {
        'q'  : 'Which year had the highest mean rainfall_mm across all grid cells?',
        'key': 'mean_rainfall_max',
        'val': max(yearly_mean_rain, key=yearly_mean_rain.get),
        'exp': f'Yearly mean rainfall ranked descending',
    },
    {
        'q'  : 'Which year had the lowest mean rainfall_mm across all grid cells?',
        'key': 'mean_rainfall_min',
        'val': min(yearly_mean_rain, key=yearly_mean_rain.get),
        'exp': f'Yearly mean rainfall ranked ascending',
    },
    {
        'q'  : 'Which year had the highest mean max_temperature_c across all grid cells?',
        'key': 'mean_temp_max',
        'val': max(yearly_mean_temp, key=yearly_mean_temp.get),
        'exp': f'Yearly mean temperature ranked descending',
    },
    {
        'q'  : 'Which year had the lowest mean max_temperature_c across all grid cells?',
        'key': 'mean_temp_min',
        'val': min(yearly_mean_temp, key=yearly_mean_temp.get),
        'exp': f'Yearly mean temperature ranked ascending',
    },
    {
        'q'  : 'Rank years from most to least anomalous by mean |anomaly_score|. Which year ranks SECOND highest?',
        'key': 'second_most_anomalous',
        'val': sorted(yearly_max_anomaly, key=yearly_max_anomaly.get, reverse=True)[1],
        'exp': f'Top 2 by max |anomaly|: {sorted(yearly_max_anomaly, key=yearly_max_anomaly.get, reverse=True)[:2]}',
    },
    {
        'q'  : 'In which two consecutive years was the change in mean anomaly_score the largest?',
        'key': 'largest_yoy_anomaly_change',
        'val': (
            lambda yrs, vals: (
                lambda diffs, idx: f'{yrs[idx]}–{yrs[idx+1]}'
            )(
                [abs(vals[y2] - vals[y1]) for y1, y2 in zip(sorted(vals)[:-1], sorted(vals)[1:])],
                int(np.argmax([abs(yearly_mean_anomaly.get(y2,0) - yearly_mean_anomaly.get(y1,0))
                               for y1, y2 in zip(years[:-1], years[1:])]))
            )
        )(years, yearly_mean_anomaly),
        'exp': 'Year-over-year absolute difference in mean anomaly_score',
    },
    {
        'q'  : 'Which year had the most balanced climate conditions (mean anomaly_score closest to zero)?',
        'key': 'most_balanced_year',
        'val': min(yearly_mean_anomaly, key=lambda y: abs(yearly_mean_anomaly[y])),
        'exp': f'Year with |mean anomaly| closest to 0',
    },
]

for q_data in global_questions:
    correct_year = q_data['val']
    other_years  = [y for y in years if y != correct_year]
    rng.shuffle(other_years)
    distractors  = list(other_years[:3])

    all_opts = [correct_year] + distractors
    rng.shuffle(all_opts)
    options  = {chr(65+i): str(v) for i, v in enumerate(all_opts)}
    ans_key  = next(k for k, v in options.items() if str(v) == str(correct_year))

    cr_counter += 1
    benchmark.append({
        'id'         : f'CR-{cr_counter:03d}',
        'category'   : 'comparative_reasoning',
        'subcategory': 'global_year_ranking',
        'difficulty' : 'medium',
        'question'   : (
            f'India-wide climate statistics per year:\n{yr_table}\n\n'
            f'{q_data["q"]}'
        ),
        'options'    : options,
        'answer'     : ans_key,
        'explanation': q_data['exp'],
        'context': {
            'yearly_mean_anomaly': {str(k): round(v,6) for k, v in yearly_mean_anomaly.items()},
            'yearly_mean_rain'   : {str(k): round(v,6) for k, v in yearly_mean_rain.items()},
            'yearly_mean_temp'   : {str(k): round(v,6) for k, v in yearly_mean_temp.items()},
            'yearly_max_anomaly' : {str(k): round(v,6) for k, v in yearly_max_anomaly.items()},
            'correct_year'       : str(correct_year),
        }
    })

print(f'Generated global year ranking questions. CR counter now: {cr_counter}')

Generated global year ranking questions. CR counter now: 40


### CR-41 to CR-50 — Best/Worst Cell in a Mini-Group (Ranking MCQ)

In [18]:
# For each question: pick 4 cells, show their max |anomaly| over all years, ask which is worst
group_idxs = rng.choice(len(cells), size=(10, 4), replace=True)

for group in group_idxs:
    group_cells = [cells[int(i)] for i in group]

    # Compute max |anomaly| for each cell across all years
    snippets = []
    for lat, lon in group_cells:
        s       = cell_series[(lat, lon)]
        max_abs = round(max(abs(v) for v in s['anomaly_score']), 4)
        max_yr  = s['years'][int(np.argmax(np.abs(s['anomaly_score'])))]
        snippets.append({
            'lat': lat, 'lon': lon,
            'max_abs_anomaly': max_abs, 'peak_year': max_yr,
        })

    ranked       = sorted(snippets, key=lambda x: x['max_abs_anomaly'], reverse=True)
    worst_cell   = ranked[0]
    correct_coord = f"({worst_cell['lat']:.2f}°N, {worst_cell['lon']:.2f}°E)"

    options = {}
    correct_key = None
    for ki, sn in enumerate(snippets):
        k = chr(65 + ki)
        options[k] = f"({sn['lat']:.2f}°N, {sn['lon']:.2f}°E) — max|anomaly|={sn['max_abs_anomaly']}"
        if sn['lat'] == worst_cell['lat'] and sn['lon'] == worst_cell['lon']:
            correct_key = k

    table_str = '\n'.join(
        f"  Cell {chr(65+ki)}: ({sn['lat']:.2f}°N, {sn['lon']:.2f}°E) — "
        f"max|anomaly|={sn['max_abs_anomaly']} in {sn['peak_year']}"
        for ki, sn in enumerate(snippets)
    )

    cr_counter += 1
    benchmark.append({
        'id'         : f'CR-{cr_counter:03d}',
        'category'   : 'comparative_reasoning',
        'subcategory': 'worst_cell_identification',
        'difficulty' : 'medium',
        'question'   : (
            f'The following grid cells are observed over {years[0]}–{years[-1]}:\n'
            f'{table_str}\n\n'
            f'Which cell had the most severe climate anomaly '
            f'(highest max|anomaly_score| across all years)?'
        ),
        'options'    : options,
        'answer'     : correct_key,
        'explanation': (
            f"Cell {correct_key} at {correct_coord} recorded the highest max|anomaly|="
            f"{worst_cell['max_abs_anomaly']} in {worst_cell['peak_year']}"
        ),
        'context': {
            'cells'      : snippets,
            'ranked'     : ranked,
            'worst_cell' : worst_cell,
        }
    })

print(f'Generated CR-041 to CR-{cr_counter:03d}')
print(f'Total comparative reasoning questions: {cr_counter}')

Generated CR-041 to CR-050
Total comparative reasoning questions: 50


## 5. Combine & Save benchmark.json

In [19]:
print(f'Trend Analysis questions    : {ta_counter}')
print(f'Comparative Reasoning q\'s  : {cr_counter}')
print(f'Total                       : {len(benchmark)}')

Trend Analysis questions    : 50
Comparative Reasoning q's  : 50
Total                       : 100


In [20]:
output = {
    'name'       : 'ClimateTwinBench',
    'version'    : '1.0',
    'description': 'Benchmark for evaluating LLM climate reasoning over India grid data (2019–2025)',
    'dataset'    : {
        'name'        : 'ClimateTwinIndia',
        'cells'       : len(cells),
        'years'       : years,
        'total_rows'  : len(df),
        'columns'     : df.columns.tolist(),
    },
    'statistics' : {
        'total_questions'     : len(benchmark),
        'by_category'         : {
            'trend_analysis'      : sum(1 for q in benchmark if q['category'] == 'trend_analysis'),
            'comparative_reasoning': sum(1 for q in benchmark if q['category'] == 'comparative_reasoning'),
        },
        'by_difficulty'       : {
            'easy'  : sum(1 for q in benchmark if q['difficulty'] == 'easy'),
            'medium': sum(1 for q in benchmark if q['difficulty'] == 'medium'),
            'hard'  : sum(1 for q in benchmark if q['difficulty'] == 'hard'),
        },
        'by_subcategory'      : dict(
            pd.Series([q['subcategory'] for q in benchmark]).value_counts()
        ),
        'class_distribution'  : {
            'LOW'     : int((df['risk_label'] == 'LOW').sum()),
            'MODERATE': int((df['risk_label'] == 'MODERATE').sum()),
            'HIGH'    : int((df['risk_label'] == 'HIGH').sum()),
            'EXTREME' : int((df['risk_label'] == 'EXTREME').sum()),
        },
        'yearly_stats'        : {
            str(y): {
                'mean_anomaly' : round(yearly_mean_anomaly[y], 6),
                'mean_rainfall': round(yearly_mean_rain[y], 6),
                'mean_temp'    : round(yearly_mean_temp[y], 6),
                'max_abs_anomaly': round(yearly_max_anomaly[y], 6),
            }
            for y in years
        },
    },
    'questions'  : benchmark,
}

with open('benchmark.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False, default=str)

print('Saved → benchmark.json')
print(f'File size: {os.path.getsize("benchmark.json") / 1024:.1f} KB')

Saved → benchmark.json
File size: 133.9 KB


## 6. Quick Verification

In [21]:
with open('benchmark.json') as f:
    loaded = json.load(f)

print(f"Total questions in file : {len(loaded['questions'])}")
print(f"By category             : {loaded['statistics']['by_category']}")
print(f"By difficulty           : {loaded['statistics']['by_difficulty']}")
print(f"By subcategory          :")
for k, v in loaded['statistics']['by_subcategory'].items():
    print(f"  {k:<45} {v}")

Total questions in file : 100
By category             : {'trend_analysis': 50, 'comparative_reasoning': 50}
By difficulty           : {'easy': 5, 'medium': 65, 'hard': 30}
By subcategory          :
  cross_cell_rainfall_comparison                15
  within_cell_year_anomaly_comparison           15
  rainfall_trend_direction                      10
  peak_anomaly_year                             10
  total_rainfall_change                         10
  multivariable_direction_agreement             10
  regime_shift_detection                        10
  global_year_ranking                           10
  worst_cell_identification                     10


In [22]:
# Print one sample from each category
for cat in ['trend_analysis', 'comparative_reasoning']:
    sample = next(q for q in loaded['questions'] if q['category'] == cat)
    print('=' * 60)
    print(f"ID       : {sample['id']}")
    print(f"Category : {sample['category']} / {sample['subcategory']}")
    print(f"Difficulty: {sample['difficulty']}")
    print(f"Question :\n{sample['question']}")
    print(f"Options  : {sample['options']}")
    print(f"Answer   : {sample['answer']}")
    print(f"Explain  : {sample['explanation']}")
    print()

ID       : TA-001
Category : trend_analysis / rainfall_trend_direction
Difficulty: medium
Question :
Annual rainfall_mm for grid cell (19.50Â°N, 94.25Â°E), 2019â€“2025:
3.31, 4.51, 2.21, 3.54, 3.56, 4.35, 4.43

Which best describes the rainfall trend?
Options  : {'A': 'Increasing (positive slope, statistically significant)', 'B': 'Decreasing (negative slope, statistically significant)', 'C': 'No significant trend (p > 0.05)'}
Answer   : C
Explain  : Slope=0.1563 but p=0.3587 > 0.05 â€” not significant

ID       : CR-001
Category : comparative_reasoning / cross_cell_rainfall_comparison
Difficulty: medium
Question :
In 2019, which grid cell had higher rainfall_mm?

Cell A: (22.75Â°N, 72.00Â°E) â€” rainfall_mm = 2.787
Cell B: (30.50Â°N, 78.00Â°E) â€” rainfall_mm = 1.709
Options  : {'A': 'Cell A (22.75Â°N, 72.00Â°E)', 'B': 'Cell B (30.50Â°N, 78.00Â°E)', 'C': 'They are approximately equal'}
Answer   : A
Explain  : Cell A=2.787 > Cell B=1.709 (diff=1.078)

